# PA4 Task 2 — Kaggle runner (exact attention, MQAR sparse/linear attention, KV-cache compression)

One Kaggle version (**Save Version → Save & Run All (Commit)**, ≤ 12 h):

| GPU 0 | GPU 1 |
|---|---|
| Part B: MQAR pools → 14 configurations × 3 seeds (4,500 updates each) → test-time mask/top-k controls → inference sweep | Part A: attention correctness + benchmark → Part C: KV-cache compression on Qwen2.5-1.5B-Instruct (180 prompts) + packed-cache benchmark |

Then the report (tables + figures) and a zip. Every step skips finished work: to resume, add this notebook's output as input,
set `PREV_OUTPUT`, and run again. Settings: **GPU T4 ×2**, **Internet On**.

In [ ]:
# ---------------- settings ----------------
REPO_URL    = "https://github.com/wasaymj/ATML.git"
BRANCH      = "main"
PREV_OUTPUT = ""          # e.g. "/kaggle/input/pa4-task2-runner" to resume

import os, subprocess, sys, time, shutil, glob, json
WORK = "/kaggle/working"
REPO = f"{WORK}/ATML"
TASK = f"{REPO}/PA4/Task 2"
DATA = f"{REPO}/PA4/atml-pa4-datasets"
LOGS = f"{WORK}/logs"
CACHE, CKPT = f"{WORK}/mqar_cache", f"{WORK}/mqar_checkpoints"
os.makedirs(LOGS, exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{TASK}/requirements.txt"], check=True)
print(subprocess.run([sys.executable, "-c", "import torch, transformers; print(torch.__version__, transformers.__version__)"],
                     capture_output=True, text=True).stdout)

In [ ]:
# ---------------- resume: restore previous outputs ----------------
def copy_tree(src, dst):
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True); print("restored", src)
if PREV_OUTPUT:
    copy_tree(f"{PREV_OUTPUT}/ATML/PA4/Task 2/results", f"{TASK}/results")
    copy_tree(f"{PREV_OUTPUT}/mqar_cache", CACHE)
    copy_tree(f"{PREV_OUTPUT}/mqar_checkpoints", CKPT)

## Model revision lock
Part C uses the Qwen2.5-1.5B-Instruct commit pinned in `atml-pa4-datasets/data/local_source_revisions.json`
(committed after the Task 1 run). If the file is missing, the same resolver as `setup_data.py` writes it
(no dataset download is needed for Task 2: MQAR and the KV prompts are already in the repository).

In [ ]:
lock = f"{DATA}/data/local_source_revisions.json"
if not os.path.exists(lock):
    subprocess.run([sys.executable, "-c", "import sys; sys.path.insert(0, 'scripts'); import setup_data; setup_data.revisions()"],
                   cwd=DATA, check=True)
rev = json.load(open(lock))["revisions"]
print(json.dumps(rev, indent=1))
from huggingface_hub import snapshot_download
print(snapshot_download("Qwen/Qwen2.5-1.5B-Instruct", revision=rev["model_target"]))
subprocess.run([sys.executable, "scripts/validate_assets.py"], cwd=DATA, check=True)   # MQAR + KV structural checks

In [ ]:
ENV = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false",
           HF_HUB_DISABLE_PROGRESS_BARS="1")
COMMON = f'--cache_dir "{CACHE}" --ckpt_dir "{CKPT}"'

def launch(name, gpu, steps):
    script = f"{LOGS}/{name}.sh"
    lines = [f'cd "{TASK}"'] + [f'{sys.executable} scripts/run_task2.py --step {s} {COMMON} || echo "STEP FAILED: {s}"' for s in steps]
    open(script, "w").write("\n".join(lines) + "\n")
    print(f"[{name}] GPU {gpu}: {steps} -> {LOGS}/{name}.log", flush=True)
    return subprocess.Popen(["bash", script], stdout=open(f"{LOGS}/{name}.log", "a"), stderr=subprocess.STDOUT,
                            env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu)))

def wait(procs, every_min=10):
    while any(p.poll() is None for p in procs.values()):
        time.sleep(every_min * 60)
        for name in procs:
            tail = open(f"{LOGS}/{name}.log", errors="replace").read().splitlines()[-3:]
            print(f"--- {name} ({time.strftime('%H:%M')}):", *tail, sep="\n   ", flush=True)
    print({n: p.returncode for n, p in procs.items()})

## Main runs

In [ ]:
import torch
if torch.cuda.device_count() >= 2:
    wait({"gpu0_partB": launch("gpu0_partB", 0, ["part_b"]),
          "gpu1_partAC": launch("gpu1_partAC", 1, ["part_a", "part_c"])})
else:
    wait({"gpu0_all": launch("gpu0_all", 0, ["part_a", "part_c", "part_b"])})
for f in glob.glob(f"{LOGS}/*.log"):
    print(f, "<-- contains a failed step" if "STEP FAILED" in open(f, errors="replace").read() else "ok")

In [ ]:
subprocess.run([sys.executable, "scripts/run_task2.py", "--step", "status"], cwd=TASK)
subprocess.run([sys.executable, "scripts/run_task2.py", "--step", "report"], cwd=TASK)
shutil.copytree(LOGS, f"{TASK}/results/logs", dirs_exist_ok=True)
shutil.make_archive(f"{WORK}/task2_results", "zip", TASK, "results")
if os.path.isdir(CKPT) and os.listdir(CKPT):          # absent if Part B produced no checkpoint
    shutil.make_archive(f"{WORK}/task2_mqar_checkpoints", "zip", CKPT)
print(sorted(os.listdir(WORK)))

In [ ]:
from IPython.display import Markdown, Image, display
for t in sorted(glob.glob(f"{TASK}/results/tables/*.md")):
    display(Markdown(f"### {os.path.basename(t)}\n" + open(t, encoding="utf-8").read()))
for fig in sorted(glob.glob(f"{TASK}/results/figures/*.png")):
    display(Image(fig, width=900))